# Project 2: Streaming Lakehouse Pipeline

NYC taxi streaming data pipeline using Spark Structured Streaming and Apache Iceberg.
Implements medallion architecture (bronze â†’ silver â†’ gold layers).

This notebook processes raw taxi events from Kafka and builds the lakehouse layers.
- **Bronze**: Raw streaming data ingestion with schema evolution support
- **Silver**: Data cleaning and enrichment
- **Gold**: Aggregation

## Initialize Spark with Iceberg & Kafka

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from datetime import datetime

# Packages are pre-loaded via PYSPARK_SUBMIT_ARGS in compose.yml
spark = (
    SparkSession.builder
    .appName("project2-pipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # Iceberg configuration
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   |  Catalog: lakehouse")

Spark 4.1.0   |  Catalog: lakehouse


## Create Database

In [2]:
try:
    spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

    # Verify the namespace exists in the Iceberg catalog
    db_exists = (
        spark.sql("SHOW NAMESPACES IN lakehouse")
        .filter("namespace = 'taxi'")
        .count() > 0
    )

    if db_exists:
        print("SUCCESS: Database lakehouse.taxi exists in catalog lakehouse")
    else:
        print("WARNING: CREATE ran, but lakehouse.taxi was not found in SHOW NAMESPACES")

except Exception as e:
    print("FAILED: Could not create or verify database lakehouse.taxi")
    print(f"Error: {e}")
    raise

SUCCESS: Database lakehouse.taxi exists in catalog lakehouse


## BRONZE LAYER: Stream Ingestion

Read raw taxi events from Kafka and write to bronze table.
Handles schema evolution (messages with/without `airport_fee` field coexist).

### Read from Kafka

In [3]:
# This notebook runs inside the Docker container, so use the internal Kafka hostname
# The producer (produce.py) defaults to localhost:9094 for host-mode execution
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

# Read streaming data from Kafka
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

print(f"Kafka source configured")
print(f"  Bootstrap: {BOOTSTRAP}")
print(f"  Topic: {TOPIC}")

Kafka source configured
  Bootstrap: kafka:9092
  Topic: taxi-trips


### Parse JSON and Handle Schema Evolution

In [4]:
# Parse JSON values and extract all fields
# Define explicit schema for all known taxi fields (handles schema evolution)
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

schema = StructType([
    StructField("VendorID", DoubleType(), True),
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", DoubleType(), True),
    StructField("DOLocationID", DoubleType(), True),
    StructField("payment_type", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)
])

parsed_stream = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("json_value"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset")
    )
    .select(
        F.from_json(
            F.col("json_value"),
            schema
        ).alias("trip_data"),
        F.col("kafka_key"),
        F.col("kafka_timestamp"),
        F.col("kafka_partition"),
        F.col("kafka_offset")
    )
    .select(
        F.col("trip_data.*"),
        F.col("kafka_key"),
        F.col("kafka_timestamp"),
        F.col("kafka_partition"),
        F.col("kafka_offset"),
        F.current_timestamp().alias("ingested_at")
    )
)

print("JSON parsing configured with schema evolution support")

JSON parsing configured with schema evolution support


### Write to Bronze Table

In [5]:
# Create Bronze table with schema matching parsed_stream
# Uses CREATE TABLE IF NOT EXISTS to preserve data across restarts
# Checkpoint idempotency ensures no duplicates even if stream restarts

bronze_table_sql = """
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze (
    VendorID DOUBLE,
    tpep_pickup_datetime STRING,
    tpep_dropoff_datetime STRING,
    passenger_count DOUBLE,
    trip_distance DOUBLE,
    RatecodeID DOUBLE,
    store_and_fwd_flag STRING,
    PULocationID DOUBLE,
    DOLocationID DOUBLE,
    payment_type DOUBLE,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE,
    kafka_key STRING,
    kafka_timestamp TIMESTAMP,
    kafka_partition INT,
    kafka_offset LONG,
    ingested_at TIMESTAMP
)
USING iceberg
"""

spark.sql(bronze_table_sql)
print("Bronze table created (or already exists): lakehouse.taxi.bronze")

Bronze table created (or already exists): lakehouse.taxi.bronze


In [6]:
CHECKPOINT_PATH = "/tmp/bronze_checkpoint"

# Write to Iceberg bronze table with checkpointing
try:
    bronze_query = (
        parsed_stream
        .writeStream
        .format("iceberg")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(processingTime="5 seconds")
        .toTable("lakehouse.taxi.bronze")
    )

    print("Bronze stream started")
    print("  Table: lakehouse.taxi.bronze")
    print(f"  Checkpoint: {CHECKPOINT_PATH}")
    print("  Trigger: 5 seconds")
    print(f"  isActive: {bronze_query.isActive}")
    print(f"  status: {bronze_query.status}")
except Exception as e:
    print("FAILED to start bronze stream")
    print(f"Error: {e}")
    raise

Bronze stream started
  Table: lakehouse.taxi.bronze
  Checkpoint: /tmp/bronze_checkpoint
  Trigger: 5 seconds
  isActive: True
  status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


### Monitor Bronze Table Growth

In [7]:
import time

print("\nMonitoring bronze table...")
print("="*80)

for interval in range(10):
    time.sleep(15)
    
    try:
        bronze_df = spark.read.table("lakehouse.taxi.bronze")
        total_rows = bronze_df.count()
        
        # Count messages with and without airport_fee
        with_fee = bronze_df.filter(F.col("airport_fee").isNotNull()).count()
        without_fee = bronze_df.filter(F.col("airport_fee").isNull()).count()
        
        elapsed = (interval + 1) * 15
        print(f"[{elapsed:>3}s] Total: {total_rows:>7}  |  "
              f"with airport_fee: {with_fee:>7}  |  "
              f"without: {without_fee:>7}")
        
        if total_rows > 1000:
            print("\nSufficient data collected (>1000 rows)")
            break
    except Exception as e:
        print(f"[{elapsed:>3}s] Waiting for table...")


Monitoring bronze table...
[ 15s] Total:  101746  |  with airport_fee:  101246  |  without:     500

Sufficient data collected (>1000 rows)


### Sample Raw Data (Both Message Shapes)

In [8]:
bronze_df = spark.read.table("lakehouse.taxi.bronze")

print("\n" + "="*80)
print("Messages WITHOUT airport_fee")
print("="*80)
(
    bronze_df
    .filter(F.col("airport_fee").isNull())
    .select(
        "VendorID", "tpep_pickup_datetime", 
        "PULocationID", "DOLocationID", 
        "fare_amount", "airport_fee"
    )
    .limit(3)
    .show(truncate=False)
)

print("\n" + "="*80)
print("Messages WITH airport_fee (rows > 500)")
print("="*80)
(
    bronze_df
    .filter(F.col("airport_fee").isNotNull())
    .select(
        "VendorID", "tpep_pickup_datetime", 
        "PULocationID", "DOLocationID", 
        "fare_amount", "airport_fee"
    )
    .limit(3)
    .show(truncate=False)
)


Messages WITHOUT airport_fee
+--------+--------------------+------------+------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|PULocationID|DOLocationID|fare_amount|airport_fee|
+--------+--------------------+------------+------------+-----------+-----------+
|1.0     |2025-01-01T00:18:38 |229.0       |237.0       |10.0       |NULL       |
|1.0     |2025-01-01T00:32:40 |236.0       |237.0       |5.1        |NULL       |
|1.0     |2025-01-01T00:44:04 |141.0       |141.0       |5.1        |NULL       |
+--------+--------------------+------------+------------+-----------+-----------+


Messages WITH airport_fee (rows > 500)
+--------+--------------------+------------+------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|PULocationID|DOLocationID|fare_amount|airport_fee|
+--------+--------------------+------------+------------+-----------+-----------+
|1.0     |2025-01-01T00:03:10 |170.0       |236.0       |14.9       |1.75       |
|1.0     |2025-01-01T00:55:

### Verify Row Counts (No Data Loss)

In [9]:
total_count = bronze_df.count()
with_fee = bronze_df.filter(F.col("airport_fee").isNotNull()).count()
without_fee = bronze_df.filter(F.col("airport_fee").isNull()).count()

print("\n" + "="*80)
print("ROW COUNT VERIFICATION")
print("="*80)
print(f"Total rows in bronze table:     {total_count:>12,}")
print(f"  WITH airport_fee (>500):      {with_fee:>12,}")
print(f"  WITHOUT airport_fee (<=500): {without_fee:>12,}")
print(f"\nExpected per file: ~3.5M rows")
print(f"Expected total: ~3.5M x 2 = ~7M rows")
print(f"\nBoth message shapes stored without data loss.")


ROW COUNT VERIFICATION
Total rows in bronze table:          108,475
  WITH airport_fee (>500):           107,975
  WITHOUT airport_fee (<=500):          500

Expected per file: ~3.5M rows
Expected total: ~3.5M x 2 = ~7M rows

Both message shapes stored without data loss.


### Restart Proof (Checkpoint Idempotency)

**PREREQUISITE:** Before running this cell, stop the producer to ensure no new messages arrive during the test.

From your host PowerShell/terminal, press `Ctrl+C` to stop `python produce.py`.

This test verifies that restarting the stream from checkpoint does not re-process old messages.

In [10]:
# Count rows BEFORE restart
count_before = spark.read.table("lakehouse.taxi.bronze").count()
print(f"Row count BEFORE restart: {count_before:,}")

# Stop stream
print("\nStopping stream...")
bronze_query.stop()

import time
time.sleep(2)

print("\n" + "="*80)
print("CHECKPOINT IDEMPOTENCY TEST")
print("="*80)
print("Make sure the producer is stopped (Ctrl+C on host) before continuing.")
print("Otherwise, new messages will be processed here (expected behavior, not duplicates).\n")

# Restart stream from checkpoint
print("Restarting stream from checkpoint...")
bronze_query = (
    parsed_stream
    .writeStream
    .format("iceberg")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze")
)

print("Stream restarted. Waiting 15 seconds to see if any messages are re-processed...")
time.sleep(15)

# Count rows AFTER restart
count_after = spark.read.table("lakehouse.taxi.bronze").count()
rows_added = count_after - count_before

print(f"\nRow count BEFORE: {count_before:,}")
print(f"Row count AFTER:  {count_after:,}")
print(f"Rows added:       {rows_added:,}")

if rows_added == 0:
    print("\nâœ“ IDEMPOTENT CHECKPOINT CONFIRMED:")
    print("  No rows were re-processed from old Kafka offsets.")
    print("  Producer was properly stopped; no new messages arrived.")
else:
    print(f"\nâš  NEW MESSAGES DETECTED: {rows_added:,} rows added")
    print("  This means the producer is still running.")
    print("  Stop the producer (Ctrl+C) and re-run this cell for a clean idempotency test.")

Row count BEFORE restart: 117,450

Stopping stream...

CHECKPOINT IDEMPOTENCY TEST
Make sure the producer is stopped (Ctrl+C on host) before continuing.
Otherwise, new messages will be processed here (expected behavior, not duplicates).

Restarting stream from checkpoint...
Stream restarted. Waiting 15 seconds to see if any messages are re-processed...

Row count BEFORE: 117,450
Row count AFTER:  117,450
Rows added:       0

âœ“ IDEMPOTENT CHECKPOINT CONFIRMED:
  No rows were re-processed from old Kafka offsets.
  Producer was properly stopped; no new messages arrived.


### Iceberg Snapshot History

In [11]:
spark.sql("SELECT * FROM lakehouse.taxi.bronze.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-03-26 15:18:03.694|3206078456509725473|NULL               |true               |
|2026-03-26 15:18:04.573|8926827396151298400|3206078456509725473|true               |
|2026-03-26 15:18:05.355|8448007292399830003|8926827396151298400|true               |
|2026-03-26 15:18:10.362|5862410473302609645|8448007292399830003|true               |
|2026-03-26 15:18:15.476|1987517641849506360|5862410473302609645|true               |
|2026-03-26 15:18:20.33 |7094583884490079680|1987517641849506360|true               |
|2026-03-26 15:18:25.305|5615372303322836208|7094583884490079680|true               |
|2026-03-26 15:18:30.35 |4747342578512292340|5615372303322836208|true               |
|2026-03-26 15:18:35.292|6599103556020049991|474734257

## SILVER LAYER: Data Cleaning & Enrichment

Reads from the bronze Iceberg table via Structured Streaming.
Each micro-batch is typed, cleaned, deduplicated, and enriched with zone names
before being appended to the silver Iceberg table.

**Changes from bronze:**
- ID fields (`VendorID`, `RatecodeID`, `PULocationID`, `DOLocationID`, `payment_type`, `passenger_count`) cast from `DOUBLE` to `INT`
- Datetime strings cast from `STRING` to `TIMESTAMP`
- Kafka metadata columns removed (not needed for analytics)
- Two new columns: `pickup_zone` and `dropoff_zone` from zone lookup join

### Create Silver Table

In [12]:
silver_table_sql = """
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver (
    vendor_id            INT,
    pickup_datetime      TIMESTAMP,
    dropoff_datetime     TIMESTAMP,
    passenger_count      INT,
    trip_distance        DOUBLE,
    ratecode_id          INT,
    store_and_fwd_flag   STRING,
    pu_location_id       INT,
    do_location_id       INT,
    payment_type         INT,
    fare_amount          DOUBLE,
    extra                DOUBLE,
    mta_tax              DOUBLE,
    tip_amount           DOUBLE,
    tolls_amount         DOUBLE,
    total_amount         DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee          DOUBLE,
    cbd_congestion_fee   DOUBLE,
    pickup_zone          STRING,
    dropoff_zone         STRING
)
USING iceberg
"""

spark.sql(silver_table_sql)
print("Silver table created (or already exists): lakehouse.taxi.silver")

Silver table created (or already exists): lakehouse.taxi.silver


### Load Zone Lookup (Static Broadcast)

In [13]:
ZONE_LOOKUP_PATH = "/home/jovyan/project/data/taxi_zone_lookup.parquet"

# Small static table (265 rows) â€” load once and cache in memory
zone_lookup = (
    spark.read.parquet(ZONE_LOOKUP_PATH)
    .select(
        F.col("LocationID").cast("int").alias("location_id"),
        F.col("Zone").alias("zone_name")
    )
)
zone_lookup.cache()

print(f"Zone lookup loaded: {zone_lookup.count()} zones")
zone_lookup.show(5, truncate=False)

Zone lookup loaded: 265 zones
+-----------+-----------------------+
|location_id|zone_name              |
+-----------+-----------------------+
|1          |Newark Airport         |
|2          |Jamaica Bay            |
|3          |Allerton/Pelham Gardens|
|4          |Alphabet City          |
|5          |Arden Heights          |
+-----------+-----------------------+
only showing top 5 rows


### Define Batch Processor (Type Casting + Cleaning + Enrichment)

In [14]:
from pyspark.sql.types import IntegerType

# Prepare two narrow lookup frames to avoid column name collisions on join
_pu_lookup = (
    zone_lookup
    .withColumnRenamed("location_id", "_pu_id")
    .withColumnRenamed("zone_name",   "pickup_zone")
)
_do_lookup = (
    zone_lookup
    .withColumnRenamed("location_id", "_do_id")
    .withColumnRenamed("zone_name",   "dropoff_zone")
)


def process_silver_batch(batch_df, batch_id):
    """Type-cast, clean, deduplicate, and zone-enrich one bronze micro-batch."""
    if batch_df.isEmpty():
        return

    # â”€â”€ Type casting â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    typed = (
        batch_df
        .withColumn("vendor_id",           F.col("VendorID").cast(IntegerType()))
        .withColumn("pickup_datetime",      F.to_timestamp("tpep_pickup_datetime", "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("dropoff_datetime",     F.to_timestamp("tpep_dropoff_datetime", "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("passenger_count",      F.col("passenger_count").cast(IntegerType()))
        .withColumn("trip_distance",        F.col("trip_distance").cast("double"))
        .withColumn("ratecode_id",          F.col("RatecodeID").cast(IntegerType()))
        .withColumn("pu_location_id",       F.col("PULocationID").cast(IntegerType()))
        .withColumn("do_location_id",       F.col("DOLocationID").cast(IntegerType()))
        .withColumn("payment_type",         F.col("payment_type").cast(IntegerType()))
        .withColumn("fare_amount",          F.col("fare_amount").cast("double"))
        .withColumn("extra",                F.col("extra").cast("double"))
        .withColumn("mta_tax",              F.col("mta_tax").cast("double"))
        .withColumn("tip_amount",           F.col("tip_amount").cast("double"))
        .withColumn("tolls_amount",         F.col("tolls_amount").cast("double"))
        .withColumn("total_amount",         F.col("total_amount").cast("double"))
        .withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
        .withColumn("airport_fee",          F.col("airport_fee").cast("double"))
        .withColumn("cbd_congestion_fee",   F.col("cbd_congestion_fee").cast("double"))
    )

    # â”€â”€ Cleaning rules â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    cleaned = (
        typed
        .filter(F.col("pickup_datetime").isNotNull())           # R1: no null pickup
        .filter(F.col("dropoff_datetime").isNotNull())          # R2: no null dropoff
        .filter(F.col("fare_amount") >= 0)                      # R3: no negative fares
        .filter(F.col("trip_distance") > 0)                     # R4: no zero distance
        .filter(
            F.col("passenger_count").isNotNull() &
            (F.col("passenger_count") >= 1) &
            (F.col("passenger_count") <= 8)
        )                                                        # R5: valid passenger count
        .dropDuplicates([
            "vendor_id", "pickup_datetime", "dropoff_datetime",
            "pu_location_id", "do_location_id"
        ])                                                       # R6: remove duplicates
    )

    # â”€â”€ Zone enrichment (broadcast join) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    enriched = (
        cleaned
        .join(F.broadcast(_pu_lookup), cleaned.pu_location_id == _pu_lookup._pu_id, "left")
        .drop("_pu_id")
        .join(F.broadcast(_do_lookup), cleaned.do_location_id == _do_lookup._do_id, "left")
        .drop("_do_id")
        .select(
            "vendor_id", "pickup_datetime", "dropoff_datetime",
            "passenger_count", "trip_distance", "ratecode_id",
            "store_and_fwd_flag", "pu_location_id", "do_location_id",
            "payment_type", "fare_amount", "extra", "mta_tax",
            "tip_amount", "tolls_amount", "total_amount",
            "congestion_surcharge", "airport_fee", "cbd_congestion_fee",
            "pickup_zone", "dropoff_zone"
        )
    )

    try:
        enriched.writeTo("lakehouse.taxi.silver").append()
        print(f"Batch {batch_id}: wrote {enriched.count()} rows to silver")
    except Exception as e:
        print(f"Batch {batch_id} FAILED: {e}")
        raise


print("Silver batch processor defined")

Silver batch processor defined


### Start Silver Stream

In [15]:
SILVER_CHECKPOINT_PATH = "/tmp/silver_checkpoint"

# Read all bronze data as a batch and process into silver
bronze_batch = spark.read.table("lakehouse.taxi.bronze")
process_silver_batch(bronze_batch, 0)

silver_count = spark.read.table("lakehouse.taxi.silver").count()
print(f"Silver table populated: {silver_count:,} rows")

# Start streaming query to pick up any new bronze rows going forward
silver_query = (
    spark.readStream
    .format("iceberg")
    .load("lakehouse.taxi.bronze")
    .writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", SILVER_CHECKPOINT_PATH)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Silver stream started for incremental updates")
print(f"  isActive: {silver_query.isActive}")

Batch 0: wrote 111427 rows to silver
Silver table populated: 111,427 rows
Silver stream started for incremental updates
  isActive: True


### Monitor Silver Table Growth

In [16]:
import time

print("Monitoring silver table...")
print("="*80)

for interval in range(10):
    time.sleep(15)
    try:
        silver_df = spark.read.table("lakehouse.taxi.silver")
        total_rows = silver_df.count()
        with_fee = silver_df.filter(F.col("airport_fee").isNotNull()).count()
        without_fee = silver_df.filter(F.col("airport_fee").isNull()).count()
        elapsed = (interval + 1) * 15
        print(f"[{elapsed:>3}s] Total: {total_rows:>7}  |  with airport_fee: {with_fee:>7}  |  without: {without_fee:>7}")
        if total_rows > 1000:
            print("Sufficient data collected (>1000 rows)")
            break
    except Exception as e:
        elapsed = (interval + 1) * 15
        print(f"[{elapsed:>3}s] Waiting for table...")

Monitoring silver table...
[ 15s] Total:  111427  |  with airport_fee:  110951  |  without:     476
Sufficient data collected (>1000 rows)


### Restart Proof (Checkpoint Idempotency)

**PREREQUISITE:** Stop the producer before running this cell.

Verifies that restarting the silver stream from checkpoint does not re-process old rows.

In [17]:
count_before = spark.read.table("lakehouse.taxi.silver").count()
print(f"Row count BEFORE restart: {count_before:,}")

print("Stopping silver stream...")
silver_query.stop()
time.sleep(2)

print("=" * 80)
print("CHECKPOINT IDEMPOTENCY TEST")
print("=" * 80)

print("Restarting silver stream from checkpoint...")
silver_query = (
    spark.readStream
    .format("iceberg")
    .load("lakehouse.taxi.bronze")
    .writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", SILVER_CHECKPOINT_PATH)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Stream restarted. Waiting 15 seconds...")
time.sleep(15)

count_after = spark.read.table("lakehouse.taxi.silver").count()
rows_added = count_after - count_before

print(f"Row count BEFORE: {count_before:,}")
print(f"Row count AFTER:  {count_after:,}")
print(f"Rows added:       {rows_added:,}")

if rows_added == 0:
    print("IDEMPOTENT CHECKPOINT CONFIRMED: No rows were re-processed.")
else:
    print(f"NEW MESSAGES DETECTED: {rows_added:,} rows added. Stop the producer and re-run.")

Row count BEFORE restart: 111,427
Stopping silver stream...
CHECKPOINT IDEMPOTENCY TEST
Restarting silver stream from checkpoint...
Stream restarted. Waiting 15 seconds...
Row count BEFORE: 111,427
Row count AFTER:  111,427
Rows added:       0
IDEMPOTENT CHECKPOINT CONFIRMED: No rows were re-processed.


### Iceberg Snapshot History

In [18]:
spark.sql("SELECT * FROM lakehouse.taxi.silver.history").show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-03-26 15:22:06.877|2558376329522294899|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+



## GOLD LAYER: Aggregation & Analytics

Aggregates silver data into hourly trip counts and fares per pickup zone.

### Create Gold Table

In [19]:
# create gold table partitioned by day for efficient date-range queries
gold_table_sql = """
CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
    pickup_hour    TIMESTAMP,
    pickup_zone    STRING,
    trip_count     LONG,
    avg_fare       DOUBLE,
    avg_distance   DOUBLE,
    total_revenue  DOUBLE
)
USING iceberg
PARTITIONED BY (days(pickup_hour))
"""

spark.sql(gold_table_sql)
print("Gold table created (or already exists): lakehouse.taxi.gold")

Gold table created (or already exists): lakehouse.taxi.gold


### Aggregate and Write

In [20]:
# aggregate silver into hourly stats per pickup zone
silver_df = spark.read.table("lakehouse.taxi.silver")

gold_df = (
    silver_df
    .withColumn("pickup_hour", F.date_trunc("hour", F.col("pickup_datetime")))
    .groupBy("pickup_hour", "pickup_zone")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("trip_distance").alias("avg_distance"),
        F.sum("total_amount").alias("total_revenue")
    )
)

gold_df.writeTo("lakehouse.taxi.gold").overwritePartitions()
print(f"Gold table written: {spark.read.table('lakehouse.taxi.gold').count():,} rows")

Gold table written: 3,899 rows


### Iceberg Snapshot History

In [21]:
spark.sql("SELECT * FROM lakehouse.taxi.gold.history").show(truncate=False)

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-03-26 15:24:55.653|5262274882016129340|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+



In [24]:
  spark.read.table("lakehouse.taxi.gold").filter("trip_count > 1").show(20, truncate=False)

+-------------------+-----------------------------+----------+------------------+------------------+------------------+
|pickup_hour        |pickup_zone                  |trip_count|avg_fare          |avg_distance      |total_revenue     |
+-------------------+-----------------------------+----------+------------------+------------------+------------------+
|2024-12-31 23:00:00|Greenwich Village South      |2         |21.2              |3.855             |63.3              |
|2024-12-31 23:00:00|JFK Airport                  |2         |36.85             |9.530000000000001 |100.45            |
|2024-12-31 23:00:00|Sutton Place/Turtle Bay North|2         |13.15             |2.435             |38.13             |
|2025-01-01 02:00:00|Alphabet City                |39        |20.787179487179483|3.689230769230769 |1156.04           |
|2025-01-01 13:00:00|Astoria                      |2         |27.45             |6.675000000000001 |70.82             |
|2025-01-01 17:00:00|Baisley Park       

## Stream Management

In [23]:
# Stop the bronze stream when complete
# bronze_query.stop()
# print("Bronze stream stopped.")

print("Bronze stream is running.")
print(f"To stop: bronze_query.stop()")
print(f"Checkpoint location: {CHECKPOINT_PATH}")

Bronze stream is running.
To stop: bronze_query.stop()
Checkpoint location: /tmp/bronze_checkpoint
